In [ ]:
from datetime import datetime, time, timedelta, timezone
from pathlib import Path

import numpy as np
import polars as pl

DATA_PATH = Path("../data/processed/timeseventeen_btc5m_features_v1.parquet")

N_FOLDS = 5
EMBARGO_SECONDS = 24 * 60 * 60  # 1 day either side of each test fold
HOLDOUT_DAYS = 14               # final untouched test period (scored once, at the end)

In [ ]:
# built in features_TimeSeventeen -> everything that isn't metadata / target is a feature

META_COLS = ["market_id", "market_date", "window_start_ts", "prediction_ts", "end_ts", "up_price"]
TARGET_COL = "resolution"

df_features = pl.read_parquet(DATA_PATH).sort("window_start_ts")
feature_cols = [c for c in df_features.columns if c not in META_COLS + [TARGET_COL]]
feature_cols

holdout split

last HOLDOUT_DAYS days held out, 1 day embargo between dev and holdout, CV only ever sees dev

In [ ]:
holdout_start_date = df_features["market_date"].max() - timedelta(days=HOLDOUT_DAYS - 1)
holdout_start_ts = int(datetime.combine(holdout_start_date, time(), tzinfo=timezone.utc).timestamp())

df_dev = df_features.filter(pl.col("end_ts") <= holdout_start_ts - EMBARGO_SECONDS)
df_holdout = df_features.filter(pl.col("window_start_ts") >= holdout_start_ts)

print(f"dev:     {df_dev['market_date'].min()} -> {df_dev['market_date'].max()}  ({df_dev.height:,} markets)")
print(f"holdout: {df_holdout['market_date'].min()} -> {df_holdout['market_date'].max()}  ({df_holdout.height:,} markets)")

purged k-fold w/ embargo

each market's information interval is [window_start_ts, end_ts] (features from window start, label at close). folds are contiguous day blocks. a training market is kept only if its interval ends at least `embargo_seconds` before the test span or starts at least `embargo_seconds` after it -> with embargo 0 this is plain purging (drops overlapping intervals only)

In [ ]:
class PurgedKFold:
    def __init__(self, n_splits=5, embargo_seconds=0):
        self.n_splits = n_splits
        self.embargo_seconds = embargo_seconds

    def split(self, t0, t1, days):
        t0, t1, days = np.asarray(t0), np.asarray(t1), np.asarray(days)
        for day_block in np.array_split(np.unique(days), self.n_splits):
            test_mask = np.isin(days, day_block)
            test_start, test_end = t0[test_mask].min(), t1[test_mask].max()
            clear_of_test = (
                (t1 <= test_start - self.embargo_seconds)
                | (t0 >= test_end + self.embargo_seconds)
            )
            yield np.flatnonzero(clear_of_test), np.flatnonzero(test_mask)


cv = PurgedKFold(n_splits=N_FOLDS, embargo_seconds=EMBARGO_SECONDS)
cv_splits = list(cv.split(df_dev["window_start_ts"], df_dev["end_ts"], df_dev["market_date"]))

In [ ]:
# fold summary + sanity checks (no shared markets, train/test gap >= embargo)

t0, t1 = df_dev["window_start_ts"].to_numpy(), df_dev["end_ts"].to_numpy()
dates = df_dev["market_date"]

fold_rows = []
for fold, (train_idx, test_idx) in enumerate(cv_splits):
    assert np.intersect1d(train_idx, test_idx).size == 0
    test_start, test_end = t0[test_idx].min(), t1[test_idx].max()
    train_before = t1[train_idx][t1[train_idx] <= test_start]
    train_after = t0[train_idx][t0[train_idx] >= test_end]
    if train_before.size:
        assert test_start - train_before.max() >= EMBARGO_SECONDS
    if train_after.size:
        assert train_after.min() - test_end >= EMBARGO_SECONDS

    fold_rows.append({
        "fold": fold,
        "test_start": dates.gather(test_idx).min(),
        "test_end": dates.gather(test_idx).max(),
        "n_train": train_idx.size,
        "n_test": test_idx.size,
    })

pl.DataFrame(fold_rows)

baselines per fold (the bar to beat)

naive -> predict the training fold's up_rate for every test market. price -> up_price as the forecast

In [2]:
def brier(p, y):
    return float(np.mean((p - y) ** 2))


X_dev = df_dev.select(feature_cols).to_numpy()
y_dev = df_dev["resolution"].to_numpy()
price_dev = df_dev["up_price"].to_numpy()

baseline_rows = []
for fold, (train_idx, test_idx) in enumerate(cv_splits):
    y_test = y_dev[test_idx]
    train_up_rate = y_dev[train_idx].mean()
    baseline_rows.append({
        "fold": fold,
        "naive_brier": brier(np.full(test_idx.size, train_up_rate), y_test),
        "price_brier": brier(price_dev[test_idx], y_test),
    })

df_baselines = pl.DataFrame(baseline_rows).with_columns(
    (1 - pl.col("price_brier") / pl.col("naive_brier")).alias("price_skill_vs_naive")
)
df_baselines

NameError: name 'df_dev' is not defined

cv loop

`make_model` returns a fresh unfitted estimator w/ fit / predict_proba, refit per fold so nothing fitted (scalers etc) sees the test fold. skill_vs_price > 0 means the model beats the market

In [ ]:
def cross_validate(make_model, cols):
    X = df_dev.select(cols).to_numpy()
    rows = []
    for fold, (train_idx, test_idx) in enumerate(cv_splits):
        model = make_model().fit(X[train_idx], y_dev[train_idx])
        p_test = model.predict_proba(X[test_idx])[:, 1]
        y_test = y_dev[test_idx]
        model_brier = brier(p_test, y_test)
        price_brier = brier(price_dev[test_idx], y_test)
        rows.append({
            "fold": fold,
            "model_brier": model_brier,
            "price_brier": price_brier,
            "skill_vs_price": 1 - model_brier / price_brier,
        })
    return pl.DataFrame(rows)

models

price_only -> logistic regression on up_logit alone (recalibrated market price). full -> + order-flow features. the gap between the two is what flow adds beyond the price

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def make_logreg():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))


cv_price_only = cross_validate(make_logreg, ["up_logit"])
cv_full = cross_validate(make_logreg, feature_cols)

pl.concat([
    cv_price_only.with_columns(pl.lit("logreg_price_only").alias("model")),
    cv_full.with_columns(pl.lit("logreg_full").alias("model")),
]).group_by("model").agg(pl.exclude("fold").mean())